# Ontology evidence and sampling temperature

**Question:** does client-side ontology grounding improve RDF answer accuracy?

| Condition | Available to the model |
|---|---|
| Endpoint | Question, endpoint and one generic SPARQL tool |
| RDFSolve | Question and client-backed MCP tools |
| RDFSolve + ontology | The same tools with optional OLS evidence |

The inputs are a **mined schema JSON** and **SPARQL Examples SHACL**. RDFSolve loads both.
Reference queries and answers stay outside model context. Every condition receives the same
question and output variable names. Final projection checks apply to every condition. The notebook runs the experiment; helpers live in `mcp_experiment.py` next to this notebook.

## 1. Inputs and budget

Set these two paths for another database. The schema supplies its endpoint; `RDFSOLVE_ENDPOINT`
can override it. Select question names with `RDFSOLVE_QUESTIONS` (a JSON list), or use every eligible example.

Start with a 30-repeat pilot. Use its cost and paired variance to choose the main repeat count before
running it. For one F1 cell mean, 385 repeats gives approximately ±0.05 worst-case Monte Carlo
precision at 95% under a normal approximation. Differences between conditions can require more.
More repeats do not compensate for too few or unrepresentative questions. Temperature zero can still
vary; seeds are paired, not guarantees of identical sampling streams.

In [ ]:
import itertools
import json
import os
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rdfsolve.schema_models import MinedSchema

root = Path(os.environ.get("RDFSOLVE_ROOT", Path.cwd().resolve().parents[1]))
sys.path.insert(0, str(root / "notebooks" / "mcp"))
from mcp_experiment import run_attempt, run_reference, evaluate, load_examples, rdf_tuples
from analysis import plan, report

schema_file = Path(os.environ.get("RDFSOLVE_SCHEMA", root / "notebooks/mcp/schemas/aopwikirdf.schema.json"))
examples_file = Path(os.environ.get("RDFSOLVE_EXAMPLES", root / "notebooks/mcp/examples/aopwiki.ttl"))
temperatures = json.loads(os.environ.get("RDFSOLVE_TEMPERATURES", "[0.0, 0.4, 0.8]"))
repeats = int(os.environ.get("RDFSOLVE_REPEATS", "30"))
seed_start = int(os.environ.get("RDFSOLVE_SEED_START", "1000"))
conditions = json.loads(os.environ.get("RDFSOLVE_CONDITIONS", '["endpoint", "rdfsolve", "rdfsolve_ols"]'))
max_response_tokens = 4096  # None disables the package ceiling.
total_output_tokens, request_limit = 32768, 32
attempt_seconds, reference_seconds = 1800, 300
ontology_seed = None  # Optional OLS snapshot copied into each attempt.
ontology_offline = False
cache_prompt = True
assert not ontology_offline or ontology_seed, "Supply an ontology snapshot for offline trials"
assert repeats > 0 and temperatures and len(set(temperatures)) == len(temperatures)
assert all(t >= 0 for t in temperatures)

output = Path(os.environ.get("RDFSOLVE_OUTPUT", root.parent / "logs/mcp-test" / datetime.now(timezone.utc).strftime("experiment-%Y%m%dT%H%M%S")))
output.mkdir(parents=True, exist_ok=True)
if ontology_seed:
    shutil.copyfile(ontology_seed, output / "ontology-initial.json")
    ontology_seed = output / "ontology-initial.json"
schema = MinedSchema.from_json(schema_file)
endpoint = os.environ.get("RDFSOLVE_ENDPOINT") or schema.about.endpoint
assert endpoint, "Set the schema endpoint or RDFSOLVE_ENDPOINT"

# The scores resolve the resources that answer values show (names, identifiers, pages).
from rdfsolve.sparql_helper import SparqlHelper
views_helper = SparqlHelper(endpoint, timeout=65)
def select(query):
    return views_helper.select_with_fallback(query, purpose="evaluation views")["results"]["bindings"]

# Generate one shared schema snapshot before any model trial.
navigation_limit = int(os.environ.get("RDFSOLVE_NAVIGATION_LIMIT", "0"))
if navigation_limit:
    from rdfsolve.sparql_helper import SparqlHelper
    with SparqlHelper(endpoint, timeout=30) as helper:
        schema.discover_paths(max_hops=3, max_paths_per_length=navigation_limit,
                              helper=helper, probe_limit=navigation_limit)
    schema_file = output / "navigation.schema.json"
    schema_file.write_text(json.dumps(schema.to_dict(), indent=2))
profiles = schema.navigation.paths if schema.navigation else []
display(pd.DataFrame([{"path": p.label(), "hops": len(p.steps),
                       "support": p.instance_support, "sources": p.source_count,
                       "matched": p.matched_sources} for p in profiles]))
print(f"Loaded {len(profiles)} elongated paths; snapshot: {schema_file}")


## 2. Questions and reference answers

SHACL labels/comments supply the natural-language question. Review their meaning against each reference
query when curating the input file. This score compares **sets of exact RDF tuples**, including unbound
values and literal metadata. Ordering, duplicate multiplicity and blank-node isomorphism need different
scoring rules. Unsupported examples and failed references are listed explicitly.

In [ ]:
cases, exclusions = load_examples(examples_file)
selected = json.loads(os.environ.get("RDFSOLVE_QUESTIONS", "[]"))
if selected:
    assert set(selected) <= {c["name"] for c in cases}, "Unknown or excluded question name"
    cases = [c for c in cases if c["name"] in selected]
references, eligible = {}, []
for index, case in enumerate(cases):
    try:
        rows = await run_reference(case["query"], endpoint, output / "references.jsonl",
                                   output / "references" / f"before-{index}", max_seconds=reference_seconds)
        references[case["name"]] = rows
        eligible.append(case)
    except Exception as exc:
        exclusions.append({"name": case["name"], "error": str(exc)})
cases = eligible
(output / "exclusions.json").write_text(json.dumps(exclusions, indent=2))
display(pd.DataFrame(exclusions, columns=["name", "error"]))
assert cases, "No reference query completed; inspect references.jsonl"
display(pd.DataFrame([{ "question": c["name"], "description": c["question"], "reference_rows": len(references[c["name"]])} for c in cases]))
planned = len(cases) * len(temperatures) * len(conditions) * repeats
print(f"{planned:,} attempts; ceiling {planned * attempt_seconds / 3600:,.1f} model hours")

## 3. Run matched comparisons

Local-model calls run on SLURM. The same model settings and budgets apply to every arm.
Question/seed blocks and condition/temperature order are shuffled. Each attempt has fresh agent state.
Every attempt has a private OLS cache, initially empty or copied from the same `ontology_seed`.
Earlier trials cannot enrich later trials. `ontology_offline=True` measures a supplied frozen snapshot;
the default permits live lookup. Missing evidence remains visible.
Prompt caching is fixed across arms and recorded; llama.cpp may produce numerical differences with
caching enabled. Report cached tokens and end-to-end time separately from F1.
Each attempt and reference query runs in a separate process with a wall-clock deadline, including
blocking retrieval. Timeout stops its child processes and scores the attempt as failed.

Failed and blocked model attempts score zero. Every attempt is saved as it completes, including the
final query, error and usage. Tool journals retain queries rejected during otherwise successful runs.

In [ ]:
assert os.environ.get("SLURM_JOB_ID"), "Submit local-model runs through SLURM"
rng = np.random.default_rng(20260915)
blocks = list(itertools.product(range(len(cases)), range(seed_start, seed_start + repeats)))
rng.shuffle(blocks)
metrics = []
assert not (output / "metrics.json").exists(), "Use a new output directory for each run"

In [ ]:
for case_index, seed in blocks:
    case = cases[case_index]
    arms = list(itertools.product(temperatures, conditions))
    rng.shuffle(arms)
    for temperature, condition in arms:
        directory = output / "attempts" / f"q{case_index}-s{seed}-t{temperature:g}-{condition}"
        directory.mkdir(parents=True, exist_ok=True)
        row = dict(question=case["name"], seed=seed, temperature=temperature, condition=condition,
                   state="running", f1=0.0, exact=False, directory=str(directory),
                   cache_prompt=cache_prompt, ontology_offline=ontology_offline,
                   ontology_seed=str(ontology_seed) if ontology_seed else None)
        metrics.append(row)
        (output / "metrics.json").write_text(json.dumps(metrics, indent=2))
        arguments = dict(
            output_variables=case["columns"],
            schema=str(schema_file), endpoint=endpoint, max_response_tokens=max_response_tokens,
            model_settings={"temperature": temperature, "seed": seed, "top_p": 1.0,
                            "extra_body": {"cache_prompt": cache_prompt}},
            usage_limits=dict(request_limit=request_limit, output_tokens_limit=total_output_tokens),
            ontology_seed=ontology_seed, ontology_offline=ontology_offline,
            output_dir=directory,
        )
        question = case["question"] + "\nReturn RDF tuples using output variables: " + ", ".join(case["columns"]) + "."
        try:
            answer = await run_attempt(question, condition, max_seconds=attempt_seconds, **arguments)
            row.update(state=answer["state"], query=answer["query"], error=answer["error"],
                       **evaluate(answer, references[case["name"]], case["columns"], select),
                       diagnostics=answer["diagnostics"])
        except Exception as exc:
            row.update(state="failed", error={"type": type(exc).__name__, "message": str(exc) or "Attempt timed out"})
        (output / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))
        print(f"{len(metrics)}/{planned} {case['name']} | {condition} | T={temperature:g} | {row['state']} | F1={row['f1']:.3f}", flush=True)

## 4. Accuracy, cost and failed queries

Re-run the reference queries once to detect answer changes during the experiment. Changed or failed
reference checks invalidate that question's comparison; their counts remain visible. This check cannot
prove that a live dataset was unchanged between checks.

In [ ]:
validity = {}
for index, case in enumerate(cases):
    try:
        current = await run_reference(case["query"], endpoint, output / "references.jsonl",
                                      output / "references" / f"after-{index}", max_seconds=reference_seconds)
        validity[case["name"]] = rdf_tuples(current, case["columns"]) == rdf_tuples(references[case["name"]], case["columns"])
    except Exception:
        validity[case["name"]] = False
results = pd.json_normalize(metrics)
results["reference_stable"] = results.question.map(validity)
results.to_csv(output / "results.csv", index=False)
display(pd.Series(validity, name="Reference answer unchanged").to_frame())
display(results.groupby(["condition", "state"]).size().rename("attempts").to_frame())

# Question-level analysis: estimates with two-stage bootstrap intervals, paired sign-flip
# tests with Holm's correction, intraclass correlation, costs and field substitutions.
stable = [r for r in metrics if validity.get(r["question"]) and r["state"] != "running"]
assert stable, "All reference checks failed or changed"
several = len({r["temperature"] for r in stable}) > 1
frame = pd.DataFrame([{
    "question": r["question"], "seed": r["seed"], "temperature": r["temperature"],
    "condition": r["condition"] + (f"@T{r['temperature']:g}" if several else ""),
    "complete": float(r["state"] == "complete"),
    **{f"{level}_{name}": float(r.get(f"{level}_{name}", 0.0))
       for level in ("term", "resource", "linked") for name in ("f1", "exact")},
    "substitutions": r.get("substitutions") or {},
    "requests": (r.get("diagnostics") or {}).get("usage", {}).get("requests"),
    "seconds": (r.get("diagnostics") or {}).get("wall_seconds"),
} for r in stable])
estimates, comparisons = report(frame, output / "analysis")
display(estimates)
display(comparisons)

failures = []
for row in metrics:
    for path in Path(row["directory"]).glob("*calls.jsonl"):
        for line in path.read_text().splitlines():
            call = json.loads(line)
            error = call.get("error") or call.get("result", {}).get("error")
            if error:
                failures.append({"question": row["question"], "condition": row["condition"], "tool": call.get("name"), "arguments": call.get("arguments"), "error": error})
(output / "failed-tools.json").write_text(json.dumps(failures, indent=2))
display(results[results.state != "complete"][["question", "condition", "temperature", "state"]])


## 5. Paired uncertainty and figures

Question means receive equal weight. Bootstrap samples retain paired conditions and temperatures while
resampling questions and their repeated seeds. Intervals require at least two questions and two repeats;
a smoke run only produces descriptive plots. Curated questions limit generalisation to other tasks.

Condition contrasts use Bonferroni-adjusted intervals across the three contrasts at each temperature.
These intervals describe sampling uncertainty; they do not validate the natural-language reference questions.

In [ ]:
# Power: fit the planning model to the first two conditions (outcome: resource-level exact
# match), then simulate the power of the sign-flip test for other numbers of questions and
# attempts. Variance between questions needs at least two attempts per question.
names = sorted(frame.condition.unique())
if len(names) >= 2 and frame.groupby(["question", "condition"]).size().min() >= 2:
    components, power = plan(frame, output / "analysis", names[0], names[1])
    display(components)
    display(power.pivot_table(index=["questions", "repeats"], columns="delta", values="power"))
print((output / "analysis" / "report.md").read_text())
